# Лабораторная работа № 3 — ЭТАЛОННОЕ РЕШЕНИЕ

## Потоковая обработка сенсорных событий в Spark Structured Streaming

**Дисциплина:** «Методы обработки больших данных»
**Направление:** 44.03.05 Педагогическое образование, профиль «Информатика и дополнительное образование (робототехника)»
**Этап пайплайна:** `Stream → Process` · **Лекция:** № 3 «Потоковые данные и распределённая обработка событий»
**Среда:** Google Colab, PySpark 4.2.0, встроенный источник `rate` · **Максимум:** 10 баллов

```text
Sense → Collect → Stream → Store → Process → Learn → Teach
                  ▲▲▲▲▲▲             ▲▲▲▲▲▲▲
```

### Критерии оценки (10 баллов)

| Часть | Содержание | Баллы |
|---|---|---:|
| 1 | Поток и телеметрия с `event_time`, опоздавшие события по варианту | 2 |
| 2 | Окна, агрегаты и **корректный снимок результата** (без дублей режима `update`) | 2 |
| 3 | Эксперимент с watermark: два значения и измеримая метрика отброшенных событий | 2 |
| 4 | Перенос на Kafka: что меняется в коде, роль key, partition, offset, checkpoint | 1 |
| 5 | **Педагогическая часть:** `TEACH CARD` (1,5) и фрагмент занятия с проверкой понимания (1,5) | 3 |

Половина инженерной и вся методическая часть опираются на **числа своего варианта**: в
работе 25 вариантов, параметры вычисляются формулой в первой ячейке.

> **Эталонный прогон — вариант 13.** Все числа в пояснениях получены при выполнении этого
> ноутбука (PySpark 4.2.0, два ядра). На вашей машине они будут другими: `rate`-источник
> зависит от производительности, а значит, и число событий в окнах тоже. Сопоставляйте
> порядок величин, а не точные значения.

### Как устроен ноутбук

Перед каждой кодовой ячейкой — блок **«Что делает ячейка»**, после — блок **«Что получено»** с
разбором чисел. Блоки **«Преподавателю»** содержат вопросы на защите и признаки типичных
ошибок. Студенческая версия (`lab_03.ipynb`) повторяет эту структуру, но вместо готового кода
содержит задания `TODO` и поля для ввода.

---

# Теоретический минимум

Полное изложение — в лекции № 3; здесь только то, без чего ячейки ниже выглядят набором
вызовов API.

## 1. Поток как неограниченная таблица

Structured Streaming представляет поток как таблицу, в которую непрерывно дописываются
строки. Запрос пишется так же, как для обычного DataFrame, а движок пересчитывает результат
инкрементально — микропакетами (*micro-batch*).

Для устойчивой системы нужна интенсивность обработки выше интенсивности поступления:

$$\lambda < \mu, \qquad \rho = \frac{\lambda}{\mu}.$$

При $\rho \to 1$ любой всплеск создаёт очередь и рост задержки.

## 2. Два времени

| Время | Что означает | Кто проставляет |
|---|---|---|
| `event_time` | когда физически произошло измерение | устройство |
| `processing_time` | когда событие обработано движком | Spark |

Задержка доставки: $L = t_{processing} - t_{event}$. Для воспроизводимой аналитики сенсорных
данных ось времени — **event time**: при повторном прогоне над теми же данными результат
обязан совпасть.

## 3. Окна

Скользящее окно длиной $W$ с шагом $S$:

$$W_k = [t_0 + kS,\; t_0 + kS + W).$$

Одно событие попадает примерно в

$$M \approx \left\lceil \frac{W}{S} \right\rceil$$

окон — это проверяется в ячейке 8 на фактических данных.

## 4. Watermark

$$\mathrm{WM}(t) = \max(t_{event}) - \Delta.$$

Окна, полностью оставшиеся позади watermark, финализируются, а их состояние удаляется.
Событие, пришедшее позже, уже не попадёт в агрегат — в метриках Spark оно отражается
счётчиком `numRowsDroppedByWatermark`.

| Большая $\Delta$ | Малая $\Delta$ |
|---|---|
| больше опоздавших событий учтено | меньше задержка финализации |
| больше состояния в памяти | меньше состояния |
| позже готов итог | выше риск потерять данные |

Именно этот компромисс измеряется в ячейке 7.

## 5. Состояние и контрольные точки

Оконный оператор хранит состояние вида «(робот, окно) → счётчик, сумма, максимум». При сбое
восстанавливаются позиция источника, состояние оператора и согласованность приёмника. В
учебном прогоне `checkpointLocation` не задаётся, в Kafka-версии он обязателен.

---

# Часть 1. Поток и телеметрия (2 балла)

## Ячейка 1. Вариант и параметры эксперимента

**Что делает ячейка.** Вычисляет параметры варианта формулой, а не берёт их из таблицы
вручную: группа `(n−1)//5` задаёт нагрузку, число роботов и частоту опоздавших событий,
позиция внутри группы `(n−1)%5` — геометрию окна, сдвиг опоздавших событий, дополнительную
метрику и целевую аудиторию для методической части. Так каждая пара «нагрузка — окно»
встречается ровно один раз, а выводы студентов не совпадают между собой.

In [1]:
# !pip -q install "pyspark==4.2.0"      # в Colab раскомментируйте эту строку
import json
import time
from typing import Dict, List

# =========== ЭТАЛОННЫЙ ПРОГОН ===========
STUDENT_NAME  = "Эталонное решение (преподаватель)"
STUDENT_GROUP = "образец"
VARIANT       = 13
# ========================================
assert 1 <= VARIANT <= 25, "Вариант — целое число от 1 до 25"


def variant_params(n: int) -> dict:
    """Параметры варианта по таблице задания.

    Группа вариантов (n−1)//5 задаёт нагрузку и число роботов,
    позиция внутри группы (n−1)%5 — геометрию окна, долю и сдвиг опоздавших событий.
    """
    rows_per_sec = [20, 30, 40, 50, 60][(n - 1) // 5]
    robots       = [3, 4, 5, 6, 8][(n - 1) // 5]
    late_every   = [11, 17, 19, 23, 29][(n - 1) // 5]     # каждое k-е событие опаздывает
    window_s, slide_s = [(10, 5), (20, 5), (20, 10), (30, 10), (15, 5)][(n - 1) % 5]
    late_shift_s = [15, 20, 25, 30, 35][(n - 1) % 5]      # на сколько сдвинут event_time
    extra_metric = ["vibration_rms", "temp_spread", "temp_stddev",
                    "hot_share", "vibration_p95"][(n - 1) % 5]
    audience     = ["5–7 класс", "8–9 класс", "10–11 класс",
                    "кружок робототехники", "СПО"][(n - 1) % 5]
    return {
        "rows_per_sec": rows_per_sec,
        "robots": robots,
        "window_s": window_s,
        "slide_s": slide_s,
        "late_every": late_every,
        "late_shift_s": late_shift_s,
        "watermark_short": "10 seconds",
        "watermark_long": f"{late_shift_s + 10} seconds",   # заведомо больше сдвига
        "extra_metric": extra_metric,
        "audience": audience,
    }


CFG = variant_params(VARIANT)
RUN_SECONDS = 15        # длительность одного прогона потока

print(f"Студент : {STUDENT_NAME}, группа {STUDENT_GROUP}")
print(f"Вариант : {VARIANT}")
for key, value in CFG.items():
    print(f"  {key:<16}: {value}")
print(f"\nОкон на одно событие (теория): M ≈ W/S = "
      f"{CFG['window_s']}/{CFG['slide_s']} = {CFG['window_s'] / CFG['slide_s']:.0f}")
print(f"Ожидаемая нагрузка: {CFG['rows_per_sec']} событий/с, "
      f"доля опоздавших ≈ {100 / CFG['late_every']:.1f} %")

Студент : Эталонное решение (преподаватель), группа образец
Вариант : 13
  rows_per_sec    : 40
  robots          : 5
  window_s        : 20
  slide_s         : 10
  late_every      : 19
  late_shift_s    : 25
  watermark_short : 10 seconds
  watermark_long  : 35 seconds
  extra_metric    : temp_stddev
  audience        : 10–11 класс

Окон на одно событие (теория): M ≈ W/S = 20/10 = 2
Ожидаемая нагрузка: 40 событий/с, доля опоздавших ≈ 5.3 %

**Что получено.** Вариант 13: 40 событий/с, пять роботов, окно 20 с с шагом 10 с, каждое
19-е событие приходит со сдвигом 25 с назад (около 5,3 % опоздавших), дополнительная метрика
— стандартное отклонение температуры, аудитория методической части — 10–11 класс. Короткий
watermark 10 с меньше сдвига опоздавших событий, длинный 35 с — больше: именно это даст
контраст в части 3.

**Преподавателю.** Первое, что проверяется в работе: совпадает ли `VARIANT` с номером студента
и не правил ли он параметры руками. Подгонка `late_shift_s` или watermark под «красивый»
результат обесценивает часть 3.

## Ячейка 2. Сессия Spark

**Что делает ячейка.** Поднимает локальную сессию. Два параметра важны для потоковой работы:
`local[*]` — потоку нужно минимум два потока выполнения (один занимает приёмник источника),
`spark.sql.shuffle.partitions = 4` — по умолчанию Spark создаёт 200 партиций шаффла, и на
маленьком Colab-прогоне это даёт сотни почти пустых задач и лишние секунды на каждый
micro-batch.

In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName(f"Lab03_StructuredStreaming_v{VARIANT}")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.streaming.metricsEnabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)
print("Ядер    :", spark.sparkContext.defaultParallelism)
print("Shuffle :", spark.conf.get("spark.sql.shuffle.partitions"))

Spark   : 4.2.0
Master  : local[*]
Ядер    : 1
Shuffle : 4

**Что получено.** Spark 4.2.0, локальный режим, четыре партиции шаффла.

**Преподавателю.** Вопрос на защите: почему для потокового запроса недостаточно одного
потока выполнения? Ответ: приёмник источника занимает его постоянно, и на обработку
micro-batch не остаётся ресурса — запрос стартует, но не выдаёт результатов.

## Ячейка 3. Телеметрия и отладка логики на конечных данных

**Что делает ячейка.** Описывает преобразование сырого потока в телеметрию робота: номер
робота из остатка по модулю, температура и вибрация как детерминированные функции счётчика,
а каждое `late_every`-е событие получает `event_time`, сдвинутый на `late_shift_s` секунд
назад — это имитация пакета, задержавшегося в беспроводной сети.

Логика вынесена в функцию и сначала проверяется на **конечной** таблице `spark.range`. Это
важный методический приём: ошибку в выражении удобнее искать там, где результат виден сразу,
а не в потоке, который приходится останавливать по таймеру.

In [3]:
def build_telemetry(source, cfg: dict):
    """Превратить сырой поток rate в телеметрию роботов.

    Логика одна и та же для потока и для статической таблицы, поэтому функция
    принимает DataFrame: так её можно отладить на конечных данных.
    """
    return (
        source
        .withColumn("robot_id", F.concat(F.lit("robot-"),
                                         (F.col("value") % cfg["robots"]).cast("string")))
        .withColumn("is_late", F.col("value") % cfg["late_every"] == 0)
        .withColumn("event_time",
                    F.when(F.col("is_late"),
                           F.col("timestamp") - F.expr(f"INTERVAL {cfg['late_shift_s']} SECONDS"))
                    .otherwise(F.col("timestamp")))
        .withColumn("motor_temp", F.lit(43.0) + (F.col("value") % 25).cast("double") * 0.55)
        .withColumn("vibration", F.abs(F.sin(F.col("value") / 8.0)))
        .select("robot_id", "event_time", "is_late", "motor_temp", "vibration")
    )


# Отладка логики на конечной таблице: те же выражения, но без потока.
preview_source = (spark.range(0, 12)
                  .withColumnRenamed("id", "value")
                  .withColumn("timestamp", F.current_timestamp()))
preview = build_telemetry(preview_source, CFG)
preview.show(12, truncate=False)
print("Опоздавших в примере:", preview.filter("is_late").count(), "из", preview.count())

+--------+--------------------------+-------+----------+-------------------+
|robot_id|event_time                |is_late|motor_temp|vibration          |
+--------+--------------------------+-------+----------+-------------------+
|robot-0 |2026-10-05 08:50:53.817764|true   |43.0      |0.0                |
|robot-1 |2026-10-05 08:51:18.817764|false  |43.55     |0.12467473338522769|
|robot-2 |2026-10-05 08:51:18.817764|false  |44.1      |0.24740395925452294|
|robot-3 |2026-10-05 08:51:18.817764|false  |44.65     |0.36627252908604757|
|robot-4 |2026-10-05 08:51:18.817764|false  |45.2      |0.479425538604203  |
|robot-0 |2026-10-05 08:51:18.817764|false  |45.75     |0.5850972729404622 |
|robot-1 |2026-10-05 08:51:18.817764|false  |46.3      |0.6816387600233341 |
|robot-2 |2026-10-05 08:51:18.817764|false  |46.85     |0.7675435022360271 |
|robot-3 |2026-10-05 08:51:18.817764|false  |47.4      |0.8414709848078965 |
|robot-4 |2026-10-05 08:51:18.817764|false  |47.95     |0.9022675940990952 |

**Что получено.** На двенадцати тестовых строках видно всё, что нужно: роботы распределены по
остатку от деления, температура и вибрация меняются детерминированно, одно событие из
двенадцати помечено `is_late` и получило `event_time` на 25 секунд раньше остальных.

**Как читать.** Поле `is_late` — это «правда» эксперимента: в реальной системе мы не знаем,
какое событие опоздало, а в модели знаем и можем проверить, учёл ли его движок.

**Преподавателю.** Типичные ошибки: сдвигают `timestamp` вместо отдельной колонки `event_time`
(тогда не на чем показать разницу двух времён); делают опоздание случайным через `rand()`
(прогон перестаёт быть воспроизводимым); сдвигают событие **вперёд**, в будущее — тогда
watermark его не отбросит, и эксперимент не получится.

---

# Часть 2. Окна, агрегаты и снимок результата (2 балла)

## Ячейка 4. Запуск потока с окнами

**Что делает ячейка.** Собирает оконную агрегацию (`window(event_time, W, S)` + `robot_id`),
запускает запрос в `memory`-приёмник и останавливает его по таймеру.

**Почему остановка по таймеру, а не `processAllAvailable()`.** Источник `rate` порождает
данные непрерывно, поэтому «дождаться обработки всех доступных данных» у него нельзя:
вызов может блокировать ячейку надолго. Корректный приём для учебного прогона — выдержать
заданное время и вызвать `stop()`.

Обратите внимание на дополнительный агрегат `late_events`: он считает, сколько опоздавших
событий попало в окно, и понадобится в эксперименте с watermark.

In [4]:
AGGREGATES = {
    "vibration_rms": F.sqrt(F.avg(F.pow("vibration", 2))).alias("extra_metric"),
    "temp_spread":   (F.max("motor_temp") - F.min("motor_temp")).alias("extra_metric"),
    "temp_stddev":   F.stddev_samp("motor_temp").alias("extra_metric"),
    "hot_share":     F.avg((F.col("motor_temp") > 50).cast("double")).alias("extra_metric"),
    "vibration_p95": F.percentile_approx("vibration", 0.95).alias("extra_metric"),
}


def run_stream(cfg: dict, watermark: str, seconds: int = RUN_SECONDS, name: str = "robot_windows"):
    """Запустить поток на заданное время и вернуть (снимок результата, метрики прогресса).

    Остановка по таймеру, а не processAllAvailable(): источник rate порождает данные
    непрерывно, и «дождаться всех доступных данных» у него нельзя — метод может
    блокировать выполнение надолго.
    """
    source = (spark.readStream.format("rate")
              .option("rowsPerSecond", cfg["rows_per_sec"])
              .option("numPartitions", 2)
              .load())
    telemetry = build_telemetry(source, cfg)
    windowed = (
        telemetry
        .withWatermark("event_time", watermark)
        .groupBy(F.window("event_time", f"{cfg['window_s']} seconds",
                          f"{cfg['slide_s']} seconds"), "robot_id")
        .agg(F.count("*").alias("events"),
             F.sum(F.col("is_late").cast("int")).alias("late_events"),
             F.avg("motor_temp").alias("avg_temp"),
             F.max("motor_temp").alias("max_temp"),
             AGGREGATES[cfg["extra_metric"]])
    )
    query = (windowed.writeStream
             .format("memory").queryName(name)
             .outputMode("update")
             .trigger(processingTime="1 second")
             .start())
    started = time.time()
    while time.time() - started < seconds:
        time.sleep(1)
    progress = list(query.recentProgress)
    query.stop()
    return spark.sql(f"SELECT * FROM {name}"), progress


raw_updates, progress_short = run_stream(CFG, CFG["watermark_short"], name="w_short")
print(f"Строк в memory-sink: {raw_updates.count()}")
print(f"Уникальных пар (окно, робот): "
      f"{raw_updates.select('window', 'robot_id').distinct().count()}")

Строк в memory-sink: 134
Уникальных пар (окно, робот): 29

**Что получено.** За 15 секунд прогона приёмник собрал **134 строки**, но уникальных пар
«окно — робот» среди них только **29**. Это не ошибка: в режиме `update` каждая строка — это
очередное обновление окна после micro-batch.

**Преподавателю.** Именно здесь проходит граница между «запустил пример» и «понял, что
получил». Если студент сдаёт число строк `memory`-таблицы как число окон — часть 2 не
засчитывается полностью.

## Ячейка 5. Снимок состояния окон

**Что делает ячейка.** Превращает историю обновлений в итоговую таблицу.

**Зачем это нужно.** В режиме `update` приёмник получает строку после **каждого** micro-batch,
в котором окно изменилось. Поэтому `memory`-таблица содержит историю, а не результат: одна и
та же пара «окно — робот» встречается многократно с разными значениями счётчика. Простой
`count()` по такой таблице измеряет число обновлений, а не число окон. Снимок строится
оконной функцией `ROW_NUMBER()`: для каждой пары берётся строка с максимальным числом
событий.

In [5]:
def snapshot(updates):
    """Снимок последнего состояния каждого окна.

    В режиме update memory-sink хранит ИСТОРИЮ обновлений: одна и та же пара
    (окно, робот) появляется после каждого micro-batch, в котором она изменилась.
    Итоговое состояние — строка с максимальным числом событий.
    """
    ranked = updates.withColumn(
        "rn", F.row_number().over(
            __import__("pyspark").sql.Window
            .partitionBy("window", "robot_id").orderBy(F.col("events").desc())))
    return (ranked.filter("rn = 1").drop("rn")
            .select(F.col("window.start").alias("window_start"),
                    F.col("window.end").alias("window_end"),
                    "robot_id", "events", "late_events",
                    F.round("avg_temp", 2).alias("avg_temp"),
                    F.round("max_temp", 2).alias("max_temp"),
                    F.round("extra_metric", 4).alias(CFG["extra_metric"]))
            .orderBy(F.col("window_start").desc(), "robot_id"))


result_short = snapshot(raw_updates)
result_short.show(12, truncate=False)
print("Окон в снимке:", result_short.count())
print("Событий учтено в окнах:", result_short.agg(F.sum("events")).first()[0])

+-------------------+-------------------+--------+------+-----------+--------+--------+-----------+
|window_start       |window_end         |robot_id|events|late_events|avg_temp|max_temp|temp_stddev|
+-------------------+-------------------+--------+------+-----------+--------+--------+-----------+
|2026-10-05 08:51:30|2026-10-05 08:51:50|robot-0 |46    |0          |48.56   |54.0    |3.9102     |
|2026-10-05 08:51:30|2026-10-05 08:51:50|robot-1 |47    |0          |48.99   |54.55   |3.9515     |
|2026-10-05 08:51:30|2026-10-05 08:51:50|robot-2 |47    |0          |49.42   |55.1    |3.948      |
|2026-10-05 08:51:30|2026-10-05 08:51:50|robot-3 |46    |0          |49.85   |55.65   |3.8989     |
|2026-10-05 08:51:30|2026-10-05 08:51:50|robot-4 |47    |0          |50.64   |56.2    |3.8674     |
|2026-10-05 08:51:20|2026-10-05 08:51:40|robot-0 |98    |0          |48.5    |54.0    |3.869      |
|2026-10-05 08:51:20|2026-10-05 08:51:40|robot-1 |99    |0          |48.99   |54.55   |3.8887     |


**Что получено.** Снимок из **29 окон**; суммарно в них учтено **1015 событий**. Видно
устройство скользящего окна: закрытое окно `08:51:20–08:51:40` содержит около 98 событий на
робота, а соседнее `08:51:30–08:51:50`, ещё не заполненное к моменту остановки, — около 47.
Колонка `late_events` показывает, что опоздавшие события попали только в старые окна
(например, окно `08:51:10–08:51:30`), и это ожидаемо: их `event_time` сдвинут на 25 с назад.

**Как читать агрегаты.** `avg_temp` и `max_temp` различаются примерно на 5–6 °C, а
`temp_stddev` около 3,9 — температура в модели пилообразная, поэтому стандартное отклонение
почти не зависит от окна. Если у студента `temp_stddev` близко к нулю — скорее всего, в окно
попало слишком мало событий.

**Преподавателю.** Вопрос на защите: почему у соседних окон число событий отличается вдвое?
Ответ: последнее окно к моменту остановки заполнено лишь наполовину — поток был остановлен
посреди него.

---

# Часть 3. Метрики потока и эксперимент с watermark (2 балла)

## Ячейка 6. Метрики micro-batch

**Что делает ячейка.** Сводит `query.recentProgress` к показателям из лекции: пропускная
способность, длительность батча, размер состояния и число событий, отброшенных watermark.
Последняя метрика — `stateOperators[0].numRowsDroppedByWatermark` — и есть измеримый ответ на
вопрос «сколько опоздавших событий потеряно».

In [6]:
def progress_summary(progress: List[dict]) -> dict:
    """Свести метрики micro-batch в показатели, обсуждавшиеся на лекции."""
    batches = [p for p in progress if p.get("numInputRows", 0) > 0]
    state = [p["stateOperators"][0] for p in batches if p.get("stateOperators")]
    return {
        "батчей": len(batches),
        "входных событий": sum(p["numInputRows"] for p in batches),
        "throughput, соб./с": round(sum(p["numInputRows"] for p in batches)
                                    / max(sum(p["durationMs"]["triggerExecution"]
                                              for p in batches) / 1000, 1e-9), 1),
        "средняя длительность батча, мс": round(sum(p["durationMs"]["triggerExecution"]
                                                    for p in batches) / len(batches)),
        "строк состояния (макс.)": max((s["numRowsTotal"] for s in state), default=0),
        "память состояния, КиБ": round(max((s["memoryUsedBytes"] for s in state), default=0) / 1024, 1),
        "событий отброшено watermark": sum(s.get("numRowsDroppedByWatermark", 0) for s in state),
    }


summary_short = progress_summary(progress_short)
print(f"Watermark {CFG['watermark_short']}\n")
for key, value in summary_short.items():
    print(f"  {key:<32}: {value}")

last = progress_short[-1]
print("\nГраницы времени последнего батча:")
for key, value in last["eventTime"].items():
    print(f"  {key:<12}: {value}")

Watermark 10 seconds

  батчей                          : 10
  входных событий                 : 520
  throughput, соб./с              : 52.3
  средняя длительность батча, мс  : 994
  строк состояния (макс.)         : 22
  память состояния, КиБ           : 11.2
  событий отброшено watermark     : 25

Границы времени последнего батча:
  avg         : 2026-10-05T08:51:34.390Z
  max         : 2026-10-05T08:51:36.128Z
  min         : 2026-10-05T08:51:10.503Z
  watermark   : 2026-10-05T08:51:25.128Z

**Что получено (watermark 10 с).**

| Показатель | Значение |
|---|---:|
| батчей с данными | 10 |
| входных событий | 520 |
| пропускная способность | 52,3 события/с |
| средняя длительность батча | 994 мс |
| строк состояния (максимум) | 22 |
| память состояния | 11,2 КиБ |
| **событий отброшено watermark** | **25** |

**Как читать.** Пропускная способность 52 события/с выше номинальных 40: движок успевает
разбирать накопленное, то есть $\rho < 1$ и очередь не растёт. Длительность батча 994 мс почти
равна интервалу триггера 1 с — система работает на грани, и при удвоении нагрузки появится
backlog. Это живая иллюстрация к модели потока из лекции.

Блок границ времени показывает механику watermark: максимальное событие в батче —
`08:51:36.128`, watermark — `08:51:25.128`, ровно на 10 секунд позади. Минимальное событие
батча `08:51:10.503` — это опоздавшая запись, и она отстала от watermark почти на 15 секунд,
поэтому попала в число отброшенных.

**Преподавателю.** 25 отброшенных событий при 520 входных — около 4,8 %, что близко к доле
опоздавших (5,3 %). Совпадение не случайное: при watermark 10 с и сдвиге 25 с **все**
опоздавшие события безнадёжно опаздывают. Хороший вопрос: почему отброшено не ровно столько,
сколько опоздало?

## Ячейка 7. Два значения watermark

**Что делает ячейка.** Повторяет тот же прогон с увеличенным watermark и сводит оба результата
в таблицу. Длинное значение подобрано так, чтобы заведомо превышать сдвиг опоздавших событий
варианта: сравнение должно показывать контраст, а не случайный разброс.

In [7]:
raw_long, progress_long = run_stream(CFG, CFG["watermark_long"], name="w_long")
result_long = snapshot(raw_long)
summary_long = progress_summary(progress_long)

comparison = [
    {"watermark": CFG["watermark_short"], **summary_short,
     "окон в снимке": result_short.count(),
     "опоздавших событий в окнах": int(result_short.agg(F.sum("late_events")).first()[0] or 0)},
    {"watermark": CFG["watermark_long"], **summary_long,
     "окон в снимке": result_long.count(),
     "опоздавших событий в окнах": int(result_long.agg(F.sum("late_events")).first()[0] or 0)},
]
import pandas as pd
print(pd.DataFrame(comparison).set_index("watermark").T.to_string())

watermark                       10 seconds  35 seconds
батчей                                10.0        13.0
входных событий                      520.0       520.0
throughput, соб./с                    52.3        55.4
средняя длительность батча, мс       994.0       723.0
строк состояния (макс.)               22.0        30.0
память состояния, КиБ                 11.2        14.2
событий отброшено watermark           25.0         0.0
окон в снимке                         29.0        30.0
опоздавших событий в окнах            31.0        56.0

**Что получено.**

| Показатель | watermark 10 с | watermark 35 с |
|---|---:|---:|
| входных событий | 520 | 520 |
| пропускная способность, соб./с | 52,3 | 55,4 |
| средняя длительность батча, мс | 994 | 723 |
| строк состояния (максимум) | 22 | 30 |
| память состояния, КиБ | 11,2 | 14,2 |
| **событий отброшено watermark** | **25** | **0** |
| окон в снимке | 29 | 30 |
| опоздавших событий учтено в окнах | 31 | 56 |

**Главный результат работы.** Увеличение watermark с 10 до 35 секунд полностью устранило
потерю опоздавших событий: 25 → 0, а число учтённых в окнах опоздавших записей выросло с 31
до 56. Цена — рост состояния: строк 22 → 30, памяти 11,2 → 14,2 КиБ, то есть примерно на
треть. В учебном прогоне это килобайты, но зависимость линейна по числу живых окон: при
тысяче роботов и окне 20 с рост состояния на треть — это уже вопрос памяти кластера.

**Чего в этой таблице нет.** Задержка финализации окна. При watermark 35 с окно закрывается на
25 секунд позже, и на 15-секундном прогоне это не видно. Честная формулировка вывода: «полнота
выросла, состояние выросло, задержку финализации этот прогон измерить не позволяет».

**Преподавателю.** Работы, где при увеличении watermark «ничего не изменилось», почти всегда
содержат одну из двух ошибок: длинный watermark меньше сдвига опоздавших событий, либо
сравнивается число строк `memory`-таблицы (оно зависит от числа батчей, а не от полноты
данных).

## Ячейка 8. Проверка кратности окон

**Что делает ячейка.** Сравнивает фактическое число «попаданий» событий в окна с теоретической
кратностью $M \approx W/S$ из лекции. Это дешёвая проверка того, что геометрия окна задана
верно: при `20 / 10` каждое событие должно учитываться примерно дважды.

In [8]:
events_total = summary_short["входных событий"]
rows_in_windows = result_short.agg(F.sum("events")).first()[0] or 0
multiplicity = rows_in_windows / max(events_total, 1)

print(f"Событий на входе за прогон        : {events_total}")
print(f"Суммарно учтено в окнах           : {rows_in_windows}")
print(f"Фактическая кратность M           : {multiplicity:.2f}")
print(f"Теоретическая кратность W/S       : {CFG['window_s'] / CFG['slide_s']:.0f}")
print("\nРасхождение объясняется краевыми окнами: события начала и конца прогона\n"
      "попадают не во все W/S окон, а часть окон ещё не финализирована.")

Событий на входе за прогон        : 520
Суммарно учтено в окнах           : 1015
Фактическая кратность M           : 1.95
Теоретическая кратность W/S       : 2

Расхождение объясняется краевыми окнами: события начала и конца прогона
попадают не во все W/S окон, а часть окон ещё не финализирована.

**Что получено.** На входе 520 событий, в окнах учтено 1015 попаданий, фактическая кратность
**1,95** против теоретической $W/S = 2$. Совпадение подтверждает, что геометрия окна задана
верно.

**Почему не ровно 2.** События начала и конца прогона попадают не во все окна: первое окно
открылось раньше начала потока, последнее не успело закрыться. Чем длиннее прогон, тем ближе
фактическая кратность к теоретической — это можно предложить проверить как дополнительный
эксперимент.

**Преподавателю.** Если у студента кратность около 1,0 при $W > S$ — скорее всего, вместо
скользящего окна задано tumbling (шаг не указан). Если заметно больше $W/S$ — вероятно,
агрегат считается по строкам истории обновлений, а не по снимку.

---

# Часть 4. Перенос на Kafka (1 балл)

## Ячейка 9. Что меняется в коде

**Что делает ячейка.** Печатает производственный вариант источника и перечисляет отличия.
Код не выполняется: broker в обязательной части не нужен, и ноутбук должен полностью
проходить в бесплатном Colab.

In [9]:
KAFKA_READY = False     # True — только при доступном broker (вне Colab)

kafka_code = """
raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "robot.telemetry")
    .option("startingOffsets", "latest")      # earliest — переиграть историю топика
    .option("maxOffsetsPerTrigger", 5000)     # ограничение нагрузки на micro-batch
    .load()
)

schema = "robot_id STRING, event_time TIMESTAMP, motor_temp DOUBLE, vibration DOUBLE"
telemetry = (
    raw.select(F.from_json(F.col("value").cast("string"), schema).alias("e"))
       .select("e.*")
)
# дальше — тот же код окон и watermark, что и в ячейке выше
"""
print(kafka_code)
print("Что меняется при переходе на Kafka:")
print("  источник   : rate -> kafka (bootstrap.servers, subscribe)")
print("  данные     : value типа BINARY -> from_json по схеме события")
print("  позиция    : startingOffsets / endingOffsets вместо отсчёта от старта процесса")
print("  параллелизм: число partition топика ограничивает число задач чтения")
print("  ключ       : key = robot_id гарантирует порядок событий одного робота внутри partition")
print("  надёжность : checkpointLocation обязателен — в нём хранятся offset и state")
print(f"\nKAFKA_READY = {KAFKA_READY}: код показан, но не выполняется — broker не требуется.")

raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "robot.telemetry")
    .option("startingOffsets", "latest")      # earliest — переиграть историю топика
    .option("maxOffsetsPerTrigger", 5000)     # ограничение нагрузки на micro-batch
    .load()
)

schema = "robot_id STRING, event_time TIMESTAMP, motor_temp DOUBLE, vibration DOUBLE"
telemetry = (
    raw.select(F.from_json(F.col("value").cast("string"), schema).alias("e"))
       .select("e.*")
)
# дальше — тот же код окон и watermark, что и в ячейке выше

Что меняется при переходе на Kafka:
  источник   : rate -> kafka (bootstrap.servers, subscribe)
  данные     : value типа BINARY -> from_json по схеме события
  позиция    : startingOffsets / endingOffsets вместо отсчёта от старта процесса
  параллелизм: число partition топика ограничивает число задач чтения
  ключ       : key = robot_id гарантирует порядок событий одного робота внутри partition
  надё

**Что получено.** Перечень изменений при переходе на Kafka: источник, разбор `value` из JSON
по схеме, позиционирование по offset, ограничение нагрузки на micro-batch, роль ключа и
обязательный `checkpointLocation`.

**Три тезиса, которые должны прозвучать в отчёте.**

1. **Offset — это позиция в partition, а не время.** Перемотка `startingOffsets = earliest`
   переигрывает историю топика, и результат оконной агрегации по event time при этом
   воспроизводится — в отличие от processing time.
2. **Ключ определяет порядок.** `key = robot_id` отправляет все события одного робота в одну
   partition, где порядок гарантирован. Глобального порядка между partition Kafka не даёт.
3. **Параллелизм ограничен числом partition.** Больше исполнителей, чем partition, — часть
   простаивает; это прямое следствие формулы $P_{consumer} \le P_{partition}$ из лекции.

**Преподавателю.** Вопрос на защите: что произойдёт с результатом, если при переходе на Kafka
оставить `processing_time` вместо `event_time`? Ответ: перемотка топика даст другие окна,
воспроизводимость потеряется.

## Ячейка 10. Самопроверка

**Что делает ячейка.** «Предполётный контроль» перед сдачей: проверяет, что все обязательные
артефакты получены. Это не оценка — проверки намеренно слабые и ловят пропуски, а не качество.

In [10]:
checks = {
    "Заполнены ФИО и группа":              STUDENT_NAME != "Иванов Иван Иванович",
    "Вариант в диапазоне 1..25":           1 <= VARIANT <= 25,
    "Поток сформировал телеметрию":        raw_updates.count() > 0,
    "Есть event_time и опоздавшие":        int(result_short.agg(F.sum("late_events")).first()[0] or 0) >= 0,
    "Окна и агрегаты посчитаны":           result_short.count() > 0,
    "Снимок без дублей обновлений":        result_short.count() <= raw_updates.count(),
    "Эксперимент с двумя watermark":       len(comparison) == 2,
    "Метрика состояния получена":          summary_short["строк состояния (макс.)"] > 0,
    "Кратность окон оценена":              multiplicity > 0,
    "Kafka-адаптация описана":             "from_json" in kafka_code,
}
for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")
print(f"\nВыполнено: {sum(checks.values())} из {len(checks)}")

spark.stop()
print("SparkSession остановлена")

OK   Заполнены ФИО и группа
OK   Вариант в диапазоне 1..25
OK   Поток сформировал телеметрию
OK   Есть event_time и опоздавшие
OK   Окна и агрегаты посчитаны
OK   Снимок без дублей обновлений
OK   Эксперимент с двумя watermark
OK   Метрика состояния получена
OK   Кратность окон оценена
OK   Kafka-адаптация описана

Выполнено: 10 из 10
SparkSession остановлена

**Что получено.** Все 10 проверок пройдены, сессия Spark корректно остановлена.

**Преподавателю.** Пройденная самопроверка не гарантирует баллов: она проверяет наличие
артефактов, а не качество выводов и методической части. Педагогическая часть (3 балла)
автоматически не проверяется вовсе.

---

# Часть 5. Педагогическая часть (3 балла)

Инженерный результат работы — три утверждения, которые можно объяснить школьнику без слов
«watermark» и «micro-batch»:

1. **У события есть два времени** — когда оно случилось и когда о нём узнали. Считать надо по
   первому.
2. **Ждать опоздавших — значит занимать память.** Чем дольше система ждёт опоздавшие данные,
   тем больше ей нужно помнить и тем позже готов итог.
3. **Скользящее окно считает одно событие несколько раз** — ровно $W/S$ раз, и это не ошибка,
   а свойство метода.

## 5.1. `TEACH CARD` (1,5 балла)

Паспорт педагогической адаптации инженерной задачи. Заполняется по **своему** варианту:
аудитория задана вариантом, числа берутся из своего прогона.

### Эталон `TEACH CARD` (вариант 13, аудитория 10–11 класс)

| Поле | Содержание |
|---|---|
| **Название учебного проекта** | «Сколько ждать опоздавших? Поток данных школьного робота» |
| **Целевая аудитория** | 10–11 класс, профильная информатика или проектная смена |
| **Исследовательский вопрос** | Сколько секунд система должна ждать опоздавшие измерения, чтобы не потерять данные и не переполнить память? |
| **Источник данных** | поток телеметрии робота: 40 событий/с от пяти роботов; каждое 19-е событие приходит с опозданием на 25 с |
| **Инженерная концепция, сохраняемая без упрощения** | различение времени события и времени обработки; окно как интервал по времени события; плата за ожидание опоздавших — объём хранимого состояния |
| **Что упрощается относительно университетской лабораторной** | нет Kafka, Spark и кода: поток разыгрывается карточками или моделируется в электронной таблице; вместо RMSE и стандартного отклонения — счёт событий и среднее |
| **Алгоритм действий обучающегося (4–6 шагов)** | 1) получить колоду карточек «событие: робот, время измерения»; 2) раздавать карточки по секундам, часть — с опозданием; 3) считать события по 20-секундным окнам, сдвигая окно на 10 с; 4) закрывать окно сразу или подождать $\Delta$ секунд; 5) сравнить, сколько карточек потеряно и сколько окон держали открытыми одновременно; 6) выбрать $\Delta$ и обосновать |
| **Измеримый результат / метрика** | число потерянных карточек и максимальное число одновременно открытых окон при $\Delta = 0$, 10, 35 с |
| **Критерий успешного выполнения** | ученик называет $\Delta$, при котором потерь нет, и объясняет, чем за это заплатили (больше открытых окон = больше памяти) |
| **Вариант усложнения для НТО / хакатона** | два потока с разной задержкой (Wi-Fi и радиоканал); требование «итог не позже чем через 15 с после конца окна» — найти $\Delta$, удовлетворяющий обоим ограничениям |
| **Риски и ограничения** | модель не передаёт реальный разброс задержек сети; при $\Delta$ больше длины занятия эффект ожидания не демонстрируется; нужна дисциплина со временем — иначе эксперимент не воспроизводится |

## 5.2. Фрагмент занятия (1,5 балла)

Описание фрагмента на 15–20 минут по **одному** из трёх выводов. Обязательны: действия
учеников (а не рассказ учителя), два числа своего варианта, ключевой вопрос и способ проверки
понимания.

### Эталон фрагмента (вывод 2: «ждать опоздавших — значит занимать память»)

**Класс и формат.** 10–11 класс, работа в группах по четыре человека, 20 минут.

**Материалы.** Колода из 60 карточек «событие» (робот 0–4, время измерения), секундомер,
лист с таблицей окон, маркеры трёх цветов.

**Ход.**

1. *(3 мин)* Учитель раздаёт роли: «робот» выдаёт карточки, «сеть» задерживает каждую
   девятнадцатую на 25 секунд, «аналитик» раскладывает карточки по окнам, «хронометрист»
   следит за временем. Окно — 20 секунд, шаг — 10 секунд.
2. *(5 мин)* Первый прогон: окно закрывается **сразу** по истечении 20 секунд. Опоздавшие
   карточки некуда положить — их складывают в стопку «потеряно».
3. *(5 мин)* Второй прогон: окно держат открытым ещё 35 секунд. Потерянных нет, но группа
   видит, сколько окон приходится держать открытыми одновременно.
4. *(4 мин)* Группы заполняют таблицу: потеряно карточек и максимум одновременно открытых
   окон для обоих режимов. Сравнение с числами настоящей системы: при ожидании 10 с движок
   отбросил **25 событий из 520**, при ожидании 35 с — **ни одного**, но объём хранимого
   состояния вырос с 22 до 30 строк, то есть примерно на треть.
5. *(3 мин)* Обсуждение: кто бы выбрал какой режим для системы предупреждения о перегреве
   мотора, а кто — для месячного отчёта.

**Ключевой вопрос.** «Данные о перегреве пришли через 30 секунд после аварии. Нужны они нам
или нет — и что изменится, если робот стоит в классе, а если на конвейере?»

**Проверка понимания.** Каждая группа отвечает письменно на два вопроса: сколько карточек
потеряно в каждом режиме и чем именно заплатили за отсутствие потерь. Засчитывается ответ, в
котором плата названа как «память» или «более поздний результат», а не «время работы
учеников». Ловится типичное заблуждение «данные пропали из-за плохого интернета»: в
эксперименте сеть работала, потеря возникла из-за решения системы не ждать.

## 5.3. Как это оценивается

| Критерий | Балл |
|---|---:|
| `TEACH CARD`: заполнены все поля, аудитория соответствует варианту, инженерная концепция названа корректно и не искажена | 1,5 |
| Фрагмент занятия: есть действия учеников, два числа своего варианта, ключевой вопрос и способ проверки понимания; фрагмент реализуем за 15–20 минут | 1,5 |

Не засчитывается: пересказ лекции без действий учеников; карточка с чужими (эталонными)
числами; фрагмент, в котором «watermark» объясняется термином «watermark».

---

# Методические замечания к проведению занятия

## Что должно остаться у студента после работы

| Уровень | Формулировка |
|---|---|
| Инженерный | умею построить оконную агрегацию по event time, задать watermark и измерить его влияние |
| Аналитический | понимаю, что увеличение полноты данных оплачивается памятью и задержкой |
| Методический | умею превратить это в задание для школьника без потери сути |

## Хронометраж пары (90 минут)

| Этап | Минут |
|---|---:|
| разбор лекционных понятий на примере ячеек 1–3 | 15 |
| самостоятельная работа: части 1–2 | 25 |
| эксперимент с watermark (часть 3) и обсуждение таблицы | 20 |
| Kafka-адаптация: разбор, без запуска | 10 |
| педагогическая часть: заполнение `TEACH CARD` и эскиз фрагмента | 20 |

## Пять вопросов на защите

1. Почему `memory`-таблица в режиме `update` содержит больше строк, чем окон?
2. Что показывает `numRowsDroppedByWatermark` и почему это число близко к доле опоздавших?
3. Чем заплатили за увеличение watermark — и что в этом прогоне измерить не удалось?
4. Почему одно событие учитывается примерно дважды при окне 20 с и шаге 10 с?
5. Что изменится в коде при переходе на Kafka и что останется прежним?

## Типичные ошибки студентов

| Ошибка | Признак в работе | Как разбирать |
|---|---|---|
| `processAllAvailable()` на `rate`-источнике | ячейка «висит» | у потока нет «всех доступных данных»: источник бесконечен |
| число строк `memory`-таблицы принято за число окон | «получено 134 окна» | показать историю обновлений одной пары «окно — робот» |
| длинный watermark меньше сдвига опоздавших | «изменений нет» | сверить watermark со сдвигом в параметрах варианта |
| опоздавшие события сдвинуты в будущее | отброшенных нет при любом watermark | watermark отсекает прошлое, а не будущее |
| выводы о задержке финализации по 15-секундному прогону | «итог стал готовиться позже» без измерения | честно указать, что прогон этого не показывает |
| `TEACH CARD` с терминами Spark | «ученики настраивают watermark» | потребовать формулировку без терминов движка |